# Data Quality Checks for People Data Lake

This notebook has all the project code in one place, in the same order as the repo:

1. **Config**: paths (`src/config.py`)
2. **Pipeline**: Bronze, Silver, and Gold layers (`src/pipeline.py`)
3. **Checks**: the 9 data quality checks (`qa/checks.py`)
4. **Run**: run all checks and save the results (`qa/run_all_checks.py`)
5. **Tests**: the tests (`tests/test_checks.py`)

The data is synthetic. Put `data/raw/workday_worker_extract.csv` in the same project folder as this notebook's parent folder (`notebooks/..`).

## 1. Setup and config

Same paths as `src/config.py`. The notebook works from the `notebooks/` folder or from the project root.

In [ ]:
from dataclasses import dataclass
from pathlib import Path
import csv
from datetime import datetime

import duckdb
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_FILE = ROOT / "data" / "raw" / "workday_worker_extract.csv"
RESULTS_DIR = ROOT / "qa" / "results"

print("Project root:", ROOT)
print("Raw file exists:", RAW_FILE.exists())

## 2. Pipeline: Bronze, Silver, Gold

- **Bronze**: the raw CSV, every column as text (`all_varchar = true`), nothing changed.
- **Silver**: typed columns. Empty strings become `NULL` with `NULLIF`.
- **Gold**: the latest row for each worker, using `QUALIFY ROW_NUMBER()`.

In [ ]:
def build_layers(con, raw_file=RAW_FILE):
    """Load the raw extract into three layers."""
    raw_path = Path(raw_file).as_posix()

    # Bronze: raw data, all columns kept as text
    con.execute(
        f"CREATE OR REPLACE TABLE bronze_workday AS "
        f"SELECT * FROM read_csv('{raw_path}', all_varchar = true)"
    )

    # Silver: correct data types and consistent empty values
    con.execute(
        """
        CREATE OR REPLACE TABLE silver_worker AS
        SELECT
            worker_id,
            worker_status,
            CAST(effective_date AS DATE)            AS effective_date,
            CAST(end_date AS DATE)                  AS end_date,
            NULLIF(job_code, '')                    AS job_code,
            NULLIF(manager_id, '')                  AS manager_id,
            location,
            CAST(NULLIF(hire_date, '') AS DATE)     AS hire_date,
            work_email,
            sin_masked
        FROM bronze_workday
        """
    )

    # Gold: latest state of each worker, one row per worker
    con.execute(
        """
        CREATE OR REPLACE TABLE gold_worker_current AS
        SELECT *
        FROM silver_worker
        QUALIFY ROW_NUMBER() OVER (
            PARTITION BY worker_id
            ORDER BY effective_date DESC
        ) = 1
        """
    )

## 3. Checks

Each check has a SQL query that returns the rows that break the rule. **No rows = PASS.** Rows = FAIL.

In [ ]:
@dataclass(frozen=True)
class Check:
    name: str
    description: str
    violations_sql: str


CHECKS = [
    Check(
        name="reconciliation_row_count",
        description="Bronze and Silver must have the same number of rows",
        violations_sql="""
            SELECT b.n AS bronze_rows, s.n AS silver_rows
            FROM (SELECT COUNT(*) AS n FROM bronze_workday) b,
                 (SELECT COUNT(*) AS n FROM silver_worker) s
            WHERE b.n <> s.n
        """,
    ),
    Check(
        name="duplicate_worker_effective_date",
        description="A worker must not have more than one record for the same effective date",
        violations_sql="""
            SELECT worker_id, effective_date, COUNT(*) AS occurrences
            FROM silver_worker
            GROUP BY worker_id, effective_date
            HAVING COUNT(*) > 1
        """,
    ),
    Check(
        name="overlapping_effective_periods",
        description="A worker's effective periods must not overlap",
        violations_sql="""
            SELECT a.worker_id,
                   a.effective_date,
                   a.end_date,
                   b.effective_date AS overlapping_start
            FROM silver_worker a
            JOIN silver_worker b
              ON a.worker_id = b.worker_id
             AND a.effective_date < b.effective_date
             AND a.end_date >= b.effective_date
        """,
    ),
    Check(
        name="end_before_start",
        description="End date must not be earlier than the start date",
        violations_sql="""
            SELECT worker_id, effective_date, end_date
            FROM silver_worker
            WHERE end_date < effective_date
        """,
    ),
    Check(
        name="missing_hire_date",
        description="Hire date must not be empty",
        violations_sql="""
            SELECT worker_id, effective_date
            FROM silver_worker
            WHERE hire_date IS NULL
        """,
    ),
    Check(
        name="missing_job_code",
        description="Job code must not be empty",
        violations_sql="""
            SELECT worker_id, effective_date
            FROM silver_worker
            WHERE job_code IS NULL
        """,
    ),
    Check(
        name="orphan_manager",
        description="The manager ID must exist as a worker in the file",
        violations_sql="""
            SELECT worker_id, manager_id
            FROM silver_worker
            WHERE manager_id IS NOT NULL
              AND manager_id NOT IN (SELECT worker_id FROM silver_worker)
        """,
    ),
    Check(
        name="sin_not_masked",
        description="Social Insurance Number must be masked as ***-***-NNN",
        violations_sql=r"""
            SELECT worker_id, sin_masked
            FROM silver_worker
            WHERE NOT regexp_matches(sin_masked, '^\*\*\*-\*\*\*-[0-9]{3}$')
        """,
    ),
    Check(
        name="gold_one_row_per_worker",
        description="The Gold layer must have exactly one row per worker",
        violations_sql="""
            SELECT worker_id, COUNT(*) AS rows_in_gold
            FROM gold_worker_current
            GROUP BY worker_id
            HAVING COUNT(*) > 1
        """,
    ),
]


def run_checks(con):
    """Run all checks and return the result of each one."""
    results = []
    for check in CHECKS:
        rows = con.sql(check.violations_sql).fetchall()
        results.append(
            {
                "check": check.name,
                "description": check.description,
                "status": "FAIL" if rows else "PASS",
                "violations": len(rows),
                "sample": "; ".join(str(row) for row in rows[:5]),
            }
        )
    return results

## 4. Run all checks

Same as `qa/run_all_checks.py`. In a notebook there is no command line, so `strict` is a function argument instead of a `--strict` flag. The results are saved to a CSV file in `qa/results/`.

In [ ]:
def run_all_checks(raw_file=RAW_FILE, strict=False):
    con = duckdb.connect()
    build_layers(con, raw_file)
    results = run_checks(con)

    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    output = RESULTS_DIR / f"qa_results_{datetime.now():%Y%m%d_%H%M%S}.csv"
    with output.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(results[0].keys()))
        writer.writeheader()
        writer.writerows(results)

    for r in results:
        print(f"[{r['status']}] {r['check']}: {r['violations']} violation(s)")

    failed = [r for r in results if r["status"] == "FAIL"]
    print(f"\n{len(results) - len(failed)} PASS, {len(failed)} FAIL")
    print(f"Results saved to: {output}")

    exit_code = 1 if (strict and failed) else 0
    return results, exit_code


results, exit_code = run_all_checks()
pd.DataFrame(results)[["check", "status", "violations", "description"]]

## 5. Tests

Same tests as `tests/test_checks.py`. The pytest version uses a fixture. Here each test is a plain function, so you can run the cell and see the output.

In [ ]:
EXPECTED_FAIL = [
    "duplicate_worker_effective_date",
    "overlapping_effective_periods",
    "end_before_start",
    "missing_hire_date",
    "missing_job_code",
    "orphan_manager",
    "sin_not_masked",
]

EXPECTED_PASS = [
    "reconciliation_row_count",
    "gold_one_row_per_worker",
]

RESULTS = {r["check"]: r for r in results}


def test_known_defects_are_detected():
    for name in EXPECTED_FAIL:
        assert RESULTS[name]["status"] == "FAIL", name
        assert RESULTS[name]["violations"] >= 1, name
    print("PASS: test_known_defects_are_detected")


def test_clean_checks_pass():
    for name in EXPECTED_PASS:
        assert RESULTS[name]["status"] == "PASS", name
    print("PASS: test_clean_checks_pass")


def test_every_check_is_covered():
    assert {c.name for c in CHECKS} == set(EXPECTED_FAIL) | set(EXPECTED_PASS)
    assert set(RESULTS) == {c.name for c in CHECKS}
    print("PASS: test_every_check_is_covered")


def test_clean_worker_is_not_flagged():
    # W1002 is valid data and must not appear in overlap or duplicate violations
    con = duckdb.connect()
    build_layers(con)
    rows = con.sql(
        "SELECT worker_id FROM silver_worker WHERE worker_id = 'W1002'"
    ).fetchall()
    assert len(rows) == 2
    assert "W1002" not in RESULTS["overlapping_effective_periods"]["sample"]
    assert "W1002" not in RESULTS["duplicate_worker_effective_date"]["sample"]
    print("PASS: test_clean_worker_is_not_flagged")


tests = [
    test_known_defects_are_detected,
    test_clean_checks_pass,
    test_every_check_is_covered,
    test_clean_worker_is_not_flagged,
]

for test in tests:
    test()

print(f"\n{len(tests)} test functions passed")

## 6. SQL exploration

The pipeline and every check already run in SQL (DuckDB). This section shows a few queries you can run yourself to look at the data and results.

In [ ]:
con = duckdb.connect()
build_layers(con)

# Full history of each worker, in order
con.sql("""
    SELECT worker_id, effective_date, end_date, job_code, manager_id, hire_date
    FROM silver_worker
    ORDER BY worker_id, effective_date
""").df()

In [ ]:
# Latest state per worker (Gold) with a count of records in history
con.sql("""
    SELECT g.worker_id,
           g.job_code,
           g.manager_id,
           h.history_rows
    FROM gold_worker_current g
    JOIN (
        SELECT worker_id, COUNT(*) AS history_rows
        FROM silver_worker
        GROUP BY worker_id
    ) h USING (worker_id)
    ORDER BY g.worker_id
""").df()

In [ ]:
# Summary of check outcomes, queried with SQL from the results of run_checks()
con.register("qa_results", pd.DataFrame(results))

con.sql("""
    SELECT status,
           COUNT(*)        AS checks,
           SUM(violations) AS total_violations
    FROM qa_results
    GROUP BY status
    ORDER BY status
""").df()